# Defect Formation Energy of Charged Defects

Calculate the formation energy of a defect as a function of its charge state and of the supercell size, with a multi-material DFT workflow on the Mat3ra platform. For the neutral defect in a single supercell see [Defect Formation Energy](defect_formation_energy.ipynb).

One job is created per (supercell size, charge state) pair. Each job takes **two materials, in order**:

- **[0] Defective supercell** — the pristine supercell with `DEFECT_CONFIGS` applied.
- **[1] Pristine supercell** — the defect-free supercell of the same size.

The workflow reports the formation energy of charge state $q$ with the Fermi level at the valence band maximum (VBM) of the pristine supercell,

$$E_f[X^q] = E_{\text{tot}}[X^q] - E_{\text{tot}}[\text{bulk}] - \sum_i \Delta N_i\, \mu_i + q\,E_{\text{VBM}} \quad [\text{eV}]$$

and this notebook draws it as a function of the electron chemical potential $\mu_e$ measured from the VBM:

$$E_f[X^q](\mu_e) = E_f[X^q] + q\,\mu_e, \qquad 0 \le \mu_e \le E_{\text{gap}}$$

$E_{\text{tot}}[\text{bulk}]$, $E_{\text{VBM}}$ and $E_{\text{gap}}$ come from a Total Energy and a Band Gap job on the pristine supercell; $\mu_i$ from the Standata elemental reference materials, as in [Formation Energy](formation_energy.ipynb). The charge $q$ enters as `tot_charge` in the QE `&SYSTEM` namelist and is compensated by a uniform jellium background.

Every Total Energy, Band Gap and defect job is tagged with its charge, `charge:<q>`. The pristine references run neutral, tagged `charge:0`, and the notebook names those jobs to the workflow, which reads its references from them alone, so a charged calculation on the same pristine cell is never taken for the neutral reference. The pristine cell can optionally be relaxed first (variable cell), and every supercell is then built from the relaxed cell.

No analytical finite-size correction (Freysoldt-Neugebauer-Van de Walle, Makov-Payne) is applied. Running several `SUPERCELL_SCALINGS` and extrapolating $E_f(L\to\infty)$ from the fitted image-charge terms takes its place; a single size gives the uncorrected value for that cell.

<h2 style="color:green">Usage</h2>

1. Set the pristine material, the defects, the supercell sizes and the charge states in cells 1.2 and 1.3 below.
1. Click "Run" > "Run All".
1. The pristine cell is relaxed if `RELAX_PRISTINE_MATERIAL` is set; Total Energy and Band Gap jobs are created for any pristine supercell that lacks them, then one Defect Formation Energy job per (size, charge).
1. Scroll down for the results table, the formation energy against the electron chemical potential with the stable charge states, and the finite-size extrapolation.

## Summary

1. Set up the environment and parameters.
1. Authenticate and initialize API client.
1. Configure compute.
1. Load the pristine cell, optionally relax it, build a pristine/defective supercell pair per size, resolve elemental references, and save them.
1. Configure the Defect Formation Energy workflow per charge state and the pristine reference workflows.
1. Create, submit, and monitor the pristine reference jobs and one defect job per (size, charge).
1. Retrieve results: formation energies, the dependence on the electron chemical potential, and the finite-size extrapolation.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
# Defect-free cell: a full Standata name, or a material in FOLDER or on the platform by its
# exact name or by a part of the name that only one material has.
PRISTINE_NAME = "Si, Silicon, FCC (Fd-3m) 3D (Bulk), mp-149"
VISUALIZATION_REPETITIONS = [1, 1, 1]

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "defect_formation_energy.json"
APPLICATION_NAME = "espresso"
MY_WORKFLOW_NAME = "Defect Formation Energy"

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

### 1.3. Set specific charge state and supercell size parameters

In [ ]:
# Defects placed in every supercell, in crystal coordinates of the pristine cell.
# See create_point_defect.ipynb for the full defect configuration reference.
DEFECT_CONFIGS = [
    {"type": "vacancy", "coordinate": [0.0, 0.0, 0.0], "placement_method": "closest_site"},
]

# Supercell sizes: n -> n x n x n repetitions of the pristine cell. Two or more
# sizes enable the finite-size extrapolation in section 7.3.
SUPERCELL_SCALINGS = [1]  # e.g. [2, 3]

# Net charge q of the defective supercell in units of e, one job per value.
CHARGES = [0]  # e.g. [1, 0, -1, -2, -3]

# K-grid of the pristine cell. A supercell of size n uses SCF_KGRID / n rounded, never below 1, which
# holds the k-point density roughly fixed -- until the grid bottoms out at 1, and larger supercells are
# then sampled more coarsely than smaller ones, an error the fit in 7.3 absorbs. None: the KPPRA default.
SCF_KGRID = None  # e.g. [8, 8, 8]

# True: relax the pristine cell first (variable cell), or reuse an earlier relaxation of it,
# and build every supercell from the relaxed cell.
RELAX_PRISTINE_MATERIAL = False

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable `OIDC_ACCESS_TOKEN`.

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account to work under

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create the compute configuration
### 3.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 3.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

### 3.3. Define how jobs are created
Every job is tagged with its charge; the pristine references run neutral, tagged `charge:0`. A reference is reused when a finished `charge:0` job on that pristine cell already reported the property (`total_energy`, `band_gaps`), whatever k-grid it ran with. The Defect Formation Energy workflow reads each reference from the one job the notebook names to it, so no other calculation on the same cell can stand in; the relaxation carries no charge tag and is never taken for one.

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async
from mat3ra.notebooks_utils.job import create_job


def create_job_for_materials(materials, workflow, tags=None):
    job = create_job(
        api_client=client,
        materials=materials,
        workflow=workflow,
        project_id=project_id,
        owner_id=ACCOUNT_ID,
        prefix=f"{workflow.name} {timestamp}",
        compute=compute.to_dict(),
        tags=tags,
    )
    return job[0] if isinstance(job, list) else job

## 4. Build the supercell pairs
### 4.1. Load the pristine cell

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.core.entity.material.api import load_material
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials


def load_pristine_material(name):
    """A Standata material by its full name, else one in FOLDER or on the platform (exact or unique partial name)."""
    standata_matches = [data for data in Materials.get_by_name(name) if data["name"] == name]
    if standata_matches:
        return Material.create(standata_matches[0])
    return load_material(client, FOLDER, name, ACCOUNT_ID)


pristine_material = load_pristine_material(PRISTINE_NAME)
print(f"Pristine material: {pristine_material.name} ({len(pristine_material.basis.elements.ids)} atoms)")

visualize_materials(pristine_material, repetitions=VISUALIZATION_REPETITIONS, title="Pristine material")

### 4.2. Relax the pristine cell (optional)
With `RELAX_PRISTINE_MATERIAL`, the lattice and the atomic positions of the pristine cell are relaxed with the Variable-cell Relaxation workflow, or taken from an earlier finished relaxation of the same cell, and the supercells below are built from the relaxed cell.

In [ ]:
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material
from mat3ra.notebooks_utils.core.entity.material.api import get_final_structure_for_job, get_or_create_material
from mat3ra.notebooks_utils.workflow import apply_scf_kgrid
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow

if RELAX_PRISTINE_MATERIAL:
    saved_pristine_material = Material.create(get_or_create_material(client, pristine_material, ACCOUNT_ID))
    relax_workflow_config = WorkflowStandata.filter_by_application(APPLICATION_NAME).get_by_name_first_match(
        "variable_cell_relaxation.json"
    )
    relax_workflow = Workflow.create(relax_workflow_config)
    relax_workflow.name = f"{relax_workflow.name} {saved_pristine_material.name}"
    apply_scf_kgrid(relax_workflow, SCF_KGRID, material=saved_pristine_material, unit_name="pw_vc-relax")
    relax_job = find_job_for_material(client, saved_pristine_material.id, relax_workflow.name, ACCOUNT_ID)
    if relax_job is None:
        relax_job = create_job_for_materials([saved_pristine_material], relax_workflow)
        submit_jobs(client.jobs, [relax_job["_id"]])
        await wait_for_jobs_to_finish_async(client.jobs, [relax_job["_id"]], poll_interval=POLL_INTERVAL)
    relaxed_pristine_material = get_final_structure_for_job(client, relax_job["_id"])
    relaxed_pristine_material.name = f"{pristine_material.name} relaxed"
    print(f"Lattice constant a: {pristine_material.lattice.a:.4f} Å as given, "
          f"{relaxed_pristine_material.lattice.a:.4f} Å relaxed (job {relax_job['_id']})")
    pristine_material = relaxed_pristine_material

### 4.3. Build a pristine/defective pair per supercell size
The pristine cell is scaled first and the defects are placed into each supercell, so that every size holds the same isolated defect at the same site. Defect coordinates are given in the pristine cell's crystal basis and are divided by the scaling to reach that site in the supercell.

Atom labels are stripped: they are useful for analysis notebooks but are not compatible with QE input generation.

In [ ]:
from mat3ra.made.tools.helpers import PointDefectDict, create_multiple_defects, create_supercell


def build_pristine_defective_pair(scaling):
    pristine_supercell = create_supercell(pristine_material, scaling_factor=[scaling] * 3)
    defect_dicts = [
        PointDefectDict(**{**config, "coordinate": [component / scaling for component in config["coordinate"]]})
        for config in DEFECT_CONFIGS
    ]
    defective_supercell = create_multiple_defects(material=pristine_supercell, defect_dicts=defect_dicts)
    for supercell, kind in ((pristine_supercell, "pristine"), (defective_supercell, "defective")):
        supercell.basis.set_labels_from_list([])
        supercell.name = f"{pristine_material.name} {scaling}x{scaling}x{scaling} {kind}"
    return pristine_supercell, defective_supercell


supercell_pairs = {scaling: build_pristine_defective_pair(scaling) for scaling in SUPERCELL_SCALINGS}

visualize_materials(
    [{"material": supercell, "title": supercell.name} for pair in supercell_pairs.values() for supercell in pair],
    repetitions=VISUALIZATION_REPETITIONS,
    rotation="-90x",
)

### 4.4. Resolve Standata elemental reference materials
Elemental chemical potentials come from Standata materials tagged `elemental` with `metadata.element`. Each must also have a refined `total_energy` -- this is enforced by the workflow at runtime; run [Total Energy](total_energy.ipynb) for any elemental reference that is missing one.

The chemical-potential term covers every species whose count changes, so elements are taken from the union of the pristine and defective structures.

In [ ]:
elements = sorted(
    {element for pair in supercell_pairs.values() for supercell in pair for element in supercell.basis.elements.values}
)
elemental_materials_data = client.materials.list(
    {"tags": "elemental", "metadata.element": {"$in": elements}},
)
available = {data.get("metadata", {}).get("element") for data in elemental_materials_data}
missing = sorted(set(elements) - available)
if missing:
    raise RuntimeError(
        f"Missing elemental reference material(s) for {missing}. "
        "Add them from Standata, or tag an existing material 'elemental' with metadata.element set."
    )
print(f"Resolved elemental reference materials for: {', '.join(elements)}")

### 4.5. Save the supercells to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_supercell_pairs = {
    scaling: tuple(Material.create(get_or_create_material(client, supercell, ACCOUNT_ID)) for supercell in pair)
    for scaling, pair in supercell_pairs.items()
}
for scaling, (pristine_supercell, defective_supercell) in saved_supercell_pairs.items():
    print(f"✅ n={scaling}: pristine {pristine_supercell.id} ({len(pristine_supercell.basis.elements.ids)} atoms), "
          f"defective {defective_supercell.id} ({len(defective_supercell.basis.elements.ids)} atoms)")

## 5. Configure the workflows
### 5.1. Select application

In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 5.2. Load the Defect Formation Energy workflow and apply size and charge
One workflow per (size, charge). The charge is set twice, from the same value: as `tot_charge` in the `&SYSTEM` namelist of the defective-cell SCF, and as the workflow's `CHARGE`, which adds $q\,E_{\text{VBM}}$. The workflow is also told which pristine jobs (section 6.1) to read its references from. The bare workflow is previewed below.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow
from mat3ra.notebooks_utils.workflow import apply_scf_kgrid, patch_workflow_qe_input, set_assignment_value

defect_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    WORKFLOW_SEARCH_TERM
)

# The workflow units that name the job each pristine reference is read from.
REFERENCE_JOB_FILTER_UNITS = {
    "total_energy": "assign-reference-job-filter",
    "band_gaps": "assign-reference-job-filter-vbm",
}


def get_scf_kgrid_for_supercell(scaling):
    """SCF_KGRID divided by the scaling, rounded, never below 1; None when SCF_KGRID is not set."""
    return None if SCF_KGRID is None else [max(1, round(dimension / scaling)) for dimension in SCF_KGRID]


def create_defect_workflow(scaling, charge, reference_jobs):
    """The workflow for one supercell size and charge; `reference_jobs` maps each property to its pristine job."""
    workflow = Workflow.create(defect_workflow_config)
    workflow.name = f"{MY_WORKFLOW_NAME} n={scaling} q={charge:+d}"
    if charge:
        patch_workflow_qe_input(workflow, {"system": {"tot_charge": charge}}, unit_names=["pw_scf"])
    set_assignment_value(workflow, "assign-charge", str(charge))
    for property_name, unit_name in REFERENCE_JOB_FILTER_UNITS.items():
        set_assignment_value(workflow, unit_name, str({"$in": [reference_jobs[property_name]["_id"]]}))
    defective_supercell = saved_supercell_pairs[scaling][1]
    return apply_scf_kgrid(workflow, get_scf_kgrid_for_supercell(scaling), material=defective_supercell)


visualize_workflow(Workflow.create(defect_workflow_config))

### 5.3. Load the pristine reference workflows
Each pristine supercell supplies two references: its total energy, which the workflow subtracts, and its band gap, which supplies $E_{\text{VBM}}$ and the range of $\mu_e$.

In [ ]:
pristine_workflow_configs = {
    property_name: WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(search_term)
    for property_name, search_term in {"total_energy": "total_energy.json", "band_gaps": "band_gap.json"}.items()
}
print(f"Loaded pristine reference workflows: {', '.join(pristine_workflow_configs)}")

## 6. Create and run the jobs
### 6.1. Create and submit the pristine reference jobs

In [ ]:
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material_with_property

pristine_reference_jobs = {scaling: {} for scaling in saved_supercell_pairs}
pristine_job_ids = []
for scaling, (pristine_supercell, _) in saved_supercell_pairs.items():
    kgrid = get_scf_kgrid_for_supercell(scaling)
    for property_name, workflow_config in pristine_workflow_configs.items():
        job = find_job_for_material_with_property(
            client, pristine_supercell.id, property_name, ACCOUNT_ID, tags=["charge:0"]
        )
        if job is not None:
            print(f"♻️  n={scaling}: reusing {property_name} from job {job['name']}")
        else:
            workflow = Workflow.create(workflow_config)
            workflow.name = f"{workflow.name} {pristine_supercell.name}"
            apply_scf_kgrid(workflow, kgrid, material=pristine_supercell)
            job = create_job_for_materials([pristine_supercell], workflow, ["charge:0"])
            pristine_job_ids.append(job["_id"])
            print(f"✅ n={scaling}: created a {property_name} job {job['_id']}, k-grid {kgrid or 'from KPPRA'}")
        pristine_reference_jobs[scaling][property_name] = job

if pristine_job_ids:
    submit_jobs(client.jobs, pristine_job_ids)
    print(f"✅ Submitted {len(pristine_job_ids)} pristine reference job(s).")

In [ ]:
if pristine_job_ids:
    await wait_for_jobs_to_finish_async(client.jobs, pristine_job_ids, poll_interval=POLL_INTERVAL)

### 6.2. Create the Defect Formation Energy jobs, one per size and charge

In [ ]:
import pandas as pd

job_records = []
for scaling, (pristine_supercell, defective_supercell) in saved_supercell_pairs.items():
    for charge in CHARGES:
        workflow = create_defect_workflow(scaling, charge, pristine_reference_jobs[scaling])
        # Order matters: [0] defective (computed), [1] pristine (reference).
        job = create_job_for_materials([defective_supercell, pristine_supercell], workflow, [f"charge:{charge}"])
        print(f"n={scaling} q={charge:+d}: k-grid {get_scf_kgrid_for_supercell(scaling) or 'from KPPRA'}, "
              f"job {job['_id']}")
        job_records.append({
            "scaling": scaling,
            "charge": charge,
            "atoms": len(defective_supercell.basis.elements.ids),
            "length": pristine_supercell.lattice.cell_volume ** (1 / 3),
            "job_id": job["_id"],
        })

job_ids = [record["job_id"] for record in job_records]
pd.DataFrame(job_records)

### 6.3. Submit the jobs and monitor the statuses

In [ ]:
if job_ids:
    submit_jobs(client.jobs, job_ids)
    print(f"✅ Submitted {len(job_ids)} Defect Formation Energy job(s).")

In [ ]:
if job_ids:
    await wait_for_jobs_to_finish_async(client.jobs, job_ids, poll_interval=POLL_INTERVAL)

## 7. Retrieve results
### 7.1. Formation energies
`formation_energy` is $E_f[X^q]$ at $\mu_e = 0$, as the workflow reports it. `length` is $L = V^{1/3}$ of the supercell, the length scale the image-charge terms in section 7.3 are expressed in.

In [ ]:
results_records = []
for record in job_records:
    job = client.jobs.get(record["job_id"])
    properties = client.properties.get_for_job(record["job_id"], property_name="defect_formation_energy")
    results_records.append({
        **record,
        "final_status": job.get("status"),
        "formation_energy": properties[0].get("value") if properties else None,
    })

results_df = pd.DataFrame(results_records)
successful_df = results_df[(results_df["final_status"] == "finished") & results_df["formation_energy"].notna()]
if len(successful_df) < len(results_df):
    print(f"⚠️  {len(results_df) - len(successful_df)} of {len(results_df)} job(s) returned no formation energy; "
          "they are left out of sections 7.2 and 7.3.")
results_df

### 7.2. Formation energy versus the electron chemical potential
Each charge state gives a straight line of slope $q$ over $\mu_e \in [0, E_{\text{gap}}]$; the lower envelope is the charge state the defect actually adopts, and the crossings between the lines are the charge transition levels. The largest supercell with a result for every charge state is used, with $E_{\text{gap}}$ from its own Band Gap job. The table lists each charge state's formation energy at the VBM and the range of the Fermi level $\mu_e$ in which it is the stable state.

In [ ]:
# The stable state is a comparison between charge states, so only a size with a result for every
# one of CHARGES is used. Missing data skips this section rather than raising, so that 7.3 still runs.
complete_scalings = [
    scaling for scaling, group in successful_df.groupby("scaling") if set(group["charge"]) == set(CHARGES)
]
largest_scaling = max(complete_scalings, default=None)
band_gap = None
if largest_scaling is None:
    print("⚠️  No supercell size has a result for every charge state in CHARGES: skipping 7.2.")
else:
    band_gap_job_id = pristine_reference_jobs[largest_scaling]["band_gaps"]["_id"]
    band_gaps = client.properties.get_for_job(band_gap_job_id, property_name="band_gaps")
    if band_gaps:
        band_gap = min(entry["value"] for entry in band_gaps[0]["values"])
        print(f"Pristine n={largest_scaling}: band gap {band_gap:.4f} eV")
    else:
        print(f"⚠️  The n={largest_scaling} pristine Band Gap job reported no band gap: skipping 7.2.")

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.defect_analysis import (
    get_charge_state_table,
    get_formation_energies_vs_fermi_level,
)
from mat3ra.notebooks_utils.ipython.entity.property.defect_plot import plot_formation_energies_vs_fermi_level
from mat3ra.notebooks_utils.ipython.plot._plotly import render_figure

charge_state_table = None
if band_gap is not None:
    largest_df = successful_df[successful_df["scaling"] == largest_scaling]
    formation_energies_at_vbm = dict(zip(largest_df["charge"], largest_df["formation_energy"]))
    formation_energies = get_formation_energies_vs_fermi_level(formation_energies_at_vbm, band_gap)
    title = f"Defect formation energy vs Fermi level (n={largest_scaling})"
    render_figure(plot_formation_energies_vs_fermi_level(formation_energies, title=title))
    charge_state_table = get_charge_state_table(formation_energies_at_vbm, band_gap)
charge_state_table

### 7.3. Finite-size extrapolation
The formation energy of a charged cell carries the spurious interaction of the defect with its periodic images, which falls off as $a/L + b/L^3$. Fitting the sizes in `SUPERCELL_SCALINGS` extrapolates to the isolated defect $E_\infty$ at $\mu_e = 0$, in place of an analytical correction; $b$ is only fitted with four or more sizes, since three would make the system square and the "fit" an interpolation.

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.defect_analysis import fit_finite_size
from mat3ra.notebooks_utils.ipython.entity.property.defect_plot import plot_finite_size_fits

fits = {}
for charge, group in successful_df.groupby("charge"):
    if len(group) < 2:
        print(f"q = {charge:+d}: E_f = {group['formation_energy'].iloc[0]:.4f} eV at n={group['scaling'].iloc[0]}; "
              "add sizes to SUPERCELL_SCALINGS to extrapolate to the isolated defect.")
        continue
    fits[charge] = fit_finite_size(group["length"], group["formation_energy"])

if fits:
    render_figure(
        plot_finite_size_fits(successful_df, fits, title="Finite-size extrapolation of the defect formation energy")
    )
pd.DataFrame([{"charge": charge, **fit} for charge, fit in fits.items()])